In [1]:
import sys, torch
print("进程ID:", sys.executable.split('\\')[-2])  # 应该不是 33236
print("PyTorch:", torch.__version__)
from transformers import AutoModelForCausalLM
print("✓ 成功！")

进程ID: nlp_DWH_2
PyTorch: 2.7.1+cu118


D:\Miniconda\envs\nlp_DWH_2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ 成功！


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "uer/gpt2-chinese-cluecorpussmall"
print(f"正在加载模型: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("✓ 模型加载成功！")

正在加载模型: uer/gpt2-chinese-cluecorpussmall


D:\Miniconda\envs\nlp_DWH_2\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--uer--gpt2-chinese-cluecorpussmall. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|████████████████████████████████████████████████████████████| 149/149 [00:00<00:00, 1

✓ 模型加载成功！


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "uer/gpt2-chinese-cluecorpussmall"
print(f"正在加载模型: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("✓ 模型加载成功！")

正在加载模型: uer/gpt2-chinese-cluecorpussmall


D:\Miniconda\envs\nlp_DWH_2\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--uer--gpt2-chinese-cluecorpussmall. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|████████████████████████████████████████████████████████████| 149/149 [00:00<00:00, 1

✓ 模型加载成功！


In [ ]:
"""
基于GPT-2的中文文本续写程序
使用Hugging Face的gpt2-chinese-cluecorpussmall模型
根据学号倒数第一位索引选取对应句子开头进行续写
"""

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 定义10个续写开头（索引0-9）
PROMPTS = {
    0: "如果我拥有一台时间机器",
    1: "当人类第一次踏上火星",
    2: "如果动物会说话，它们最想告诉人类的是",
    3: "有一天，城市突然停电了",
    4: "当我醒来，发现自己变成了一本书",
    5: "假如我能隐身一天，我会",
    6: "我走进了那扇从未打开过的门",
    7: "在一个没有网络的世界里",
    8: "如果世界上只剩下我一个人",
    9: "梦中醒来，一切都变了模样"
}

def load_model(model_name="uer/gpt2-chinese-cluecorpussmall"):
    """
    加载GPT-2中文模型和分词器
    """
    print(f"\n正在加载模型: {model_name}")
    print("首次运行会下载模型（约500MB），请耐心等待...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    print(f"✓ 模型加载完成，使用设备: {device}")
    return tokenizer, model, device

def generate_text(prompt, tokenizer, model, device,
                  max_length=200,
                  temperature=0.8,
                  top_k=50,
                  top_p=0.95):
    """生成续写文本"""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

def main():
    student_id = input("请输入你的学号: ").strip()

    if not student_id.isdigit():
        print("错误：学号必须是数字！")
        return

    last_digit = int(student_id[-1])
    prompt = PROMPTS[last_digit]

    print(f"\n{'='*60}")
    print(f"学号: {student_id} | 倒数第一位: {last_digit}")
    print(f"选中的开头: 「{prompt}」")
    print(f"{'='*60}")

    try:
        tokenizer, model, device = load_model()

        print("\n正在生成续写内容...\n")
        result = generate_text(prompt, tokenizer, model, device)

        print(f"{'='*60}")
        print("【续写结果】")
        print(f"{'='*60}")
        print(result)

    except Exception as e:
        print(f"\n✗ 错误: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

请输入你的学号:  2



学号: 2 | 倒数第一位: 2
选中的开头: 「如果动物会说话，它们最想告诉人类的是」

正在加载模型: uer/gpt2-chinese-cluecorpussmall
首次运行会下载模型（约500MB），请耐心等待...


Loading weights: 100%|████████████████████████████████████████████████████████████| 149/149 [00:00<00:00, 11070.29it/s]
GPT2LMHeadModel LOAD REPORT from: uer/gpt2-chinese-cluecorpussmall
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
D:\Miniconda\envs\nlp_DWH_2\lib\site-packages\torch\cuda\__init__.py:287: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


✓ 模型加载完成，使用设备: cuda

正在生成续写内容...

